<a href="https://colab.research.google.com/github/P-SAM-SAHIL/alc_newspaper_proj/blob/main/NU_Master_List.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

df = pd.read_csv('/content/chronicling-america.csv')

# 1. Convert Dates
df['First Issue'] = pd.to_datetime(df['First Issue'], errors='coerce')
df['Last Issue'] = pd.to_datetime(df['Last Issue'], errors='coerce')

# Extract Year for trend analysis
df['Start Year'] = df['First Issue'].dt.year

# 2. Split Geo Location into Latitude and Longitude
# Some locations might be NaN, so we handle them gracefully
df[['Latitude', 'Longitude']] = df['Geo Location'].str.split(',', expand=True)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# 3. Clean 'Number of Issues'
df['Number of Issues'] = pd.to_numeric(df['Number of Issues'], errors='coerce')

print("Data cleaning complete! Here's a quick look at the info:")
df.info()

Data cleaning complete! Here's a quick look at the info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4709 entries, 0 to 4708
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Newspapers               4709 non-null   object        
 1   LCCN                     4709 non-null   object        
 2   OCLC                     4709 non-null   object        
 3   ISSN                     4195 non-null   object        
 4   State                    4708 non-null   object        
 5   County                   4397 non-null   object        
 6   City                     4704 non-null   object        
 7   Geo Location             4708 non-null   object        
 8   Browse Digitized Issues  4708 non-null   object        
 9   Number of Issues         4708 non-null   float64       
 10  First Issue              4708 non-null   datetime64[ns]
 11  Last Issue               4708 non-null

In [ ]:
# Count newspapers per state
state_counts = df['State'].value_counts().reset_index()
state_counts.columns = ['State', 'Newspaper Count']

fig_state = px.bar(
    state_counts.head(20), # Top 20 states
    x='State',
    y='Newspaper Count',
    title='Top 20 States by Number of Digitized Newspapers',
    labels={'Newspaper Count': 'Number of Newspapers', 'State': 'State'},
    color='Newspaper Count',
    color_continuous_scale='Viridis'
)

fig_state.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_state.show()

In [ ]:
# Count newspapers per state
state_counts = df['State'].value_counts().reset_index()
state_counts.columns = ['State', 'Newspaper Count']

fig_state = px.bar(
    state_counts,
    x='State',
    y='Newspaper Count',
    title='All States by Number of Digitized Newspapers',
    labels={'Newspaper Count': 'Number of Newspapers', 'State': 'State'},
    color='Newspaper Count',
    color_continuous_scale='Viridis'
)

fig_state.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig_state.show()

In [ ]:
# Drop rows without coordinates for the map
map_df = df.dropna(subset=['Latitude', 'Longitude'])

fig_map = px.scatter_mapbox(
    map_df,
    lat="Latitude",
    lon="Longitude",
    hover_name="Newspapers",
    hover_data=["City", "State", "First Issue", "Last Issue"],
    color="Number of Issues",
    color_continuous_scale=px.colors.sequential.Plasma,
    size_max=15,
    zoom=3,
    title="Geographical Distribution of Newspapers",
    mapbox_style="carto-positron"
)

fig_map.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_map.show()

In [ ]:
# Group by Start Year
yearly_starts = df.groupby('Start Year').size().reset_index(name='Newspapers Started')

# Filter out bad dates (e.g., years before 1600 or in the future)
yearly_starts = yearly_starts[(yearly_starts['Start Year'] >= 1700) & (yearly_starts['Start Year'] <= 2024)]

fig_timeline = px.line(
    yearly_starts,
    x='Start Year',
    y='Newspapers Started',
    title='Timeline of Newspaper Inceptions (Based on First Issue Year)',
    markers=True,
    template='plotly_dark'
)

fig_timeline.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of New Newspapers',
    hovermode='x unified'
)
fig_timeline.show()

In [ ]:
# Get top 10 languages (excluding NaN)
lang_counts = df['Languages'].dropna().value_counts().reset_index()
lang_counts.columns = ['Language', 'Count']

# Group smaller counts into 'Other' if there are too many
top_langs = lang_counts.head(10)

fig_lang = px.pie(
    top_langs,
    names='Language',
    values='Count',
    title='Top 10 Languages of Digitized Newspapers',
    hole=0.4, # Makes it a donut chart
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig_lang.update_traces(textposition='inside', textinfo='percent+label')
fig_lang.show()

In [ ]:
# We use a log scale for the Y-axis because a few papers have tens of thousands of issues,
# while most might only have a few hundred.
fig_hist = px.histogram(
    df,
    x='Number of Issues',
    nbins=50,
    title='Distribution of Digitized Issues per Newspaper',
    color_discrete_sequence=['indianred'],
    log_y=True # Log scale helps handle extreme outliers
)

fig_hist.update_layout(
    xaxis_title='Total Number of Issues',
    yaxis_title='Count of Newspapers (Log Scale)',
    template='ggplot2'
)
fig_hist.show()

In [ ]:
# Calculate Lifespan in Years
df['Lifespan (Years)'] = (df['Last Issue'] - df['First Issue']).dt.days / 365.25

# To keep the chart readable, let's only look at the top 10 states with the most papers
top_10_states = df['State'].value_counts().nlargest(10).index
lifespan_df = df[df['State'].isin(top_10_states)].dropna(subset=['Lifespan (Years)'])

fig_lifespan = px.violin(
    lifespan_df,
    x='State',
    y='Lifespan (Years)',
    color='State',
    box=True, # Adds a boxplot inside the violin
    points="outliers", # Shows outlier points
    hover_data=['Newspapers', 'First Issue', 'Last Issue'],
    title='Distribution of Newspaper Lifespans in Top 10 States'
)

fig_lifespan.update_layout(template='plotly_white', showlegend=False)
fig_lifespan.show()

In [ ]:
# Filter out rows where Ethnicity is NaN
eth_df = df.dropna(subset=['Ethnicity'])

# Group by Ethnicity and State
eth_counts = eth_df.groupby(['Ethnicity', 'State']).size().reset_index(name='Count')

fig_eth = px.treemap(
    eth_counts,
    path=[px.Constant("All Targeted Newspapers"), 'Ethnicity', 'State'],
    values='Count',
    title='Distribution of Newspapers by Target Ethnicity and State',
    color='Count',
    color_continuous_scale='RdBu'
)

fig_eth.update_traces(root_color="lightgrey")
fig_eth.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig_eth.show()

In [ ]:
# Filter out rows where Ethnicity is NaN
eth_df = df.dropna(subset=['Ethnicity'])

# Group by Ethnicity and State
eth_counts = eth_df.groupby(['Ethnicity', 'County']).size().reset_index(name='Count')

fig_eth = px.treemap(
    eth_counts,
    path=[px.Constant("All Targeted Newspapers"), 'Ethnicity', 'County'],
    values='Count',
    title='Distribution of Newspapers by Target Ethnicity and county',
    color='Count',
    color_continuous_scale='RdBu'
)

fig_eth.update_traces(root_color="lightgrey")
fig_eth.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig_eth.show()

In [ ]:
# Count by City and sort
city_counts = df['City'].value_counts().reset_index()
city_counts.columns = ['City', 'Newspaper Count']
top_cities = city_counts.head(15).sort_values(by='Newspaper Count', ascending=True)

fig_cities = px.bar(
    top_cities,
    x='Newspaper Count',
    y='City',
    orientation='h', # Horizontal bar chart for readable labels
    title='Top 15 City Hubs for Newspaper Publishing',
    text='Newspaper Count',
    color='Newspaper Count',
    color_continuous_scale='Teal'
)

fig_cities.update_traces(textposition='outside')
fig_cities.update_layout(template='plotly_dark')
fig_cities.show()

In [ ]:
# Drop rows where 'Essay Available' or 'Number of Issues' is missing
essay_df = df.dropna(subset=['Essay Available', 'Number of Issues'])

fig_essay = px.box(
    essay_df,
    x='Essay Available',
    y='Number of Issues',
    color='Essay Available',
    log_y=True, # Using log scale again due to massive variance in issue counts
    title='Number of Digitized Issues: Essay Available vs. Not Available',
    labels={'Essay Available': 'Has Historical Essay', 'Number of Issues': 'Total Issues (Log Scale)'}
)

fig_essay.update_layout(template='ggplot2')
fig_essay.show()

In [ ]:
# --- Plotting Newspaper Lifespans (Start and End Dates) ---

# 1. Ensure dates are parsed correctly (you likely already have this from earlier)
df['First Issue'] = pd.to_datetime(df['First Issue'], errors='coerce')
df['Last Issue'] = pd.to_datetime(df['Last Issue'], errors='coerce')

# 2. Filter the data to make the chart readable
# Option A: Plot everything (only recommended for small datasets/samples)
# plot_df = df.dropna(subset=['First Issue', 'Last Issue'])

# Option B: Filter for a specific state or subset (e.g., South Carolina)
plot_df = df[df['State'] == 'South Carolina'].dropna(subset=['First Issue', 'Last Issue'])

# Option C: Just grab the top 30 longest-running papers to visualize
# plot_df = df.dropna(subset=['First Issue', 'Last Issue']).sort_values(by='Lifespan (Years)', ascending=False).head(30)

# 3. Create the Timeline (Gantt) Chart
fig_lifespan_gantt = px.timeline(
    plot_df,
    x_start="First Issue",
    x_end="Last Issue",
    y="Newspapers",
    color="State", # Color-code by state
    hover_data=["City", "Number of Issues", "First Issue", "Last Issue"],
    title="Newspaper Publication Lifespans (Start to End)",
    template='plotly_white'
)

# 4. Reverse the Y-axis so the first entry appears at the top of the chart
fig_lifespan_gantt.update_yaxes(autorange="reversed")

# Show the plot
fig_lifespan_gantt.show()

In [ ]:
# --- Detailing the Start and End Dates ---

# Create a clean summary dataframe
details_df = df[['Newspapers', 'State', 'City', 'First Issue', 'Last Issue', 'Number of Issues']].copy()

# Drop rows where we don't have exact dates
details_df = details_df.dropna(subset=['First Issue', 'Last Issue'])

# Calculate exact duration in days and years
details_df['Days Active'] = (details_df['Last Issue'] - details_df['First Issue']).dt.days
details_df['Years Active'] = round(details_df['Days Active'] / 365.25, 2)

# Sort alphabetically by Newspaper name (or change to 'First Issue' to sort chronologically)
details_df = details_df.sort_values(by='Newspapers')

print("Detailed Publication Lifespans:")
# Display the first 20 records (in Colab, removing 'print()' and just running 'details_df.head(20)' will display a nice interactive HTML table)
display(details_df.head(20))

Detailed Publication Lifespans:


,Newspapers,State,City,First Issue,Last Issue,Number of Issues,Days Active,Years Active
3,"Abbeville Press (Abbeville, S.C.) 1860-1869",South Carolina,Abbeville,1860-11-09,1869-09-24,275.0,3241,8.87
5,"Abbeville Progress (Abbeville, Vermilion Paris...",Louisiana,Abbeville,1913-03-01,1921-12-31,462.0,3227,8.84
6,"Abendblatt (Chicago, Ill.) 1894-1899",Illinois,Chicago,1894-10-24,1899-07-27,1470.0,1737,4.76
7,Abendblatt Der Illinois Staats-Zeitung (Chicag...,Illinois,Chicago,1893-03-27,1894-10-23,487.0,575,1.57
10,"Aberdeen Herald (Aberdeen, Chehalis County, W....",Washington,Aberdeen,1890-10-23,1917-06-29,1979.0,9745,26.68
13,"Abilene Weekly Reflector (Abilene, Kan.) 1888-...",Kansas,Abilene,1888-05-03,1922-12-28,1813.0,12656,34.65
15,"Abolition Standard (Concord, N.H.) 1840-1841",New Hampshire,Concord,1840-07-21,1841-01-01,2.0,164,0.45
16,"Ad-Daleel (Detroit, Mich.) 194?-19??",Michigan,Detroit,1943-12-08,1951-12-20,189.0,2934,8.03
17,Adahooniłigii (Phoenix [Ariz.]) 1943-????,Arizona,Phoenix,1943-08-02,1957-05-01,103.0,5021,13.75
21,"Adams County News (Ritzville, Wash.) 1898-1906",Washington,Ritzville,1898-02-02,1906-10-17,443.0,3178,8.70
